# Prophet – 30‑Minute Waste Forecast

This notebook trains a Prophet model for each canteen section. Prophet automatically captures daily and weekly seasonality.

Saves models and plots to `deployment_models/`.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from google.colab import drive

warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# Mount Google Drive and set the working directory
drive.mount('/content/drive', force_remount=True)
try:
    os.chdir('/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence')
    print('Directory changed to project folder')
except OSError:
    print("Error: Could not change directory. Please check the path.")

## Configuration

We set the parameters for data processing and the Prophet model. These control how the data is aggregated and how the model is trained.

In [ ]:
DATA_PATH   = 'data/food_waste_augmented.csv'
MODEL_DIR   = 'deployment_models'
MODEL_NAME  = 'Prophet'

FREQ        = '30min'
ALIGN_SECTIONS = 'union'          # 'common' drops missing, 'union' fills with 0
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.1
TEST_RATIO  = 0.2
LOOKBACK    = 24
MIN_SAMPLES = 10

os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Aggregation frequency: {FREQ}")
print(f"Model: {MODEL_NAME}")

## Load and Aggregate Data

We read the waste data, resample it to 30‑minute intervals, and sum the waste per canteen section. Then we pivot the table so that each section becomes a separate column.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
daily_section = (
    df.set_index('Date')
      .groupby([pd.Grouper(freq=FREQ), 'Canteen_Section'])['Waste_Weight_kg']
      .sum()
      .reset_index()
      .rename(columns={'Waste_Weight_kg': 'Total_Waste_kg'})
)

In [ ]:
daily_wide = (
    daily_section
    .pivot(index='Date', columns='Canteen_Section', values='Total_Waste_kg')
    .sort_index()
)

if ALIGN_SECTIONS == 'common':
    daily_wide = daily_wide.dropna()
else:
    daily_wide = daily_wide.fillna(0)

sections = daily_wide.columns.tolist()
print(f"Sections: {sections}")

## Split into Train, Validation, Test

We divide the time series chronologically. The first 70% is for training, next 10% for validation, and the last 20% for testing.

In [ ]:
ref_index = daily_wide.index
n_total   = len(ref_index)
n_test = max(1, int(n_total * TEST_RATIO))
n_val  = max(1, int(n_total * VAL_RATIO))
n_train = n_total - n_val - n_test

train_indices = ref_index[:n_train]
val_indices   = ref_index[n_train:n_train + n_val]
test_indices  = ref_index[n_train + n_val:]

train_mask = ref_index.isin(train_indices)
val_mask   = ref_index.isin(val_indices)
test_mask  = ref_index.isin(test_indices)

print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

## Helper Functions

We define a function to compute regression metrics and another to plot predictions.

In [ ]:
def compute_regression_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R²."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

In [ ]:
def plot_predictions(y_true, y_pred, dates, section, max_points=200, save_path=None):
    """Plot predicted vs actual waste over time."""
    if len(dates) > max_points:
        step = len(dates) // max_points
        idx = range(0, len(dates), step)
        dates = dates[idx]; y_true = y_true[idx]; y_pred = y_pred[idx]
    plt.figure(figsize=(12,5))
    plt.plot(dates, y_true, label='Actual', marker='o', markersize=3, linestyle='-', linewidth=1)
    plt.plot(dates, y_pred, label='Predicted', marker='x', markersize=3, linestyle='--', linewidth=1)
    plt.xlabel('Date'); plt.ylabel('Waste (kg)')
    plt.title(f'Prophet Predictions vs Actual - Section {section}')
    plt.legend(); plt.grid(True); plt.xticks(rotation=45); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## Train a Separate Prophet Model for Each Canteen Section

We loop through each section, prepare the data in the format Prophet expects (a DataFrame with columns `ds` and `y`), train the model, and evaluate it on the test set.

In [ ]:
all_metrics = []

In [ ]:
for sec in sections:
    print(f"\n{'='*60}\nTraining Prophet for section {sec}\n{'='*60}")

    y_series = daily_wide[sec]
    y_train = y_series[train_mask]
    y_test  = y_series[test_mask]

    if len(y_train) == 0:
        print(f"  Skipping section {sec} – training set empty.")
        continue

    # Prepare data for Prophet
    train_df = pd.DataFrame({'ds': y_train.index, 'y': y_train.values})
    test_df  = pd.DataFrame({'ds': test_indices})

In [ ]:
# Train Prophet
model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False
)
model.fit(train_df)

# Forecast
forecast = model.predict(test_df)
test_pred = forecast['yhat'].values

In [ ]:
# Metrics
metrics = compute_regression_metrics(y_test.values, test_pred)
print("\n--- Performance ---")
for k,v in metrics.items(): print(f"  {k}: {v:.4f}")

all_metrics.append({'Section': sec, 'Model': MODEL_NAME, **metrics})

In [ ]:
# Save model and metadata
artifacts = {
    'section': sec,
    'model_name': MODEL_NAME,
    'model': model,
    'train_date_range': [train_indices.min().isoformat(), train_indices.max().isoformat()],
    'val_date_range': [val_indices.min().isoformat(), val_indices.max().isoformat()],
    'test_date_range': [test_indices.min().isoformat(), test_indices.max().isoformat()]
}
save_path = f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}.joblib'
joblib.dump(artifacts, save_path)
print(f"Saved model to {save_path}")

In [ ]:
# Plot predictions
plot_predictions(y_test.values, test_pred, test_indices, sec,
                    save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_predictions.png')

## Save Summary of Results

After training all sections, we collect the performance metrics into a CSV file for later comparison.

In [ ]:
if all_metrics:
    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(f'{MODEL_DIR}/test_metrics_{MODEL_NAME}.csv', index=False)
    print("\nSummary saved.")
else:
    print("No metrics collected.")